# Example: Using the Data Pipeline for Analysis

This notebook demonstrates how to use `dlt-ibapi`'s data readers to perform analysis on market data.

**Educational Examples** - These are simple demonstrations of data pipeline patterns, not production trading strategies.

## Table of Contents

1. [Calculate Option Metrics](#1-calculate-option-metrics)
2. [Find Liquid Contracts](#2-find-liquid-contracts)
3. [Analyze Volume Patterns](#3-analyze-volume-patterns)
4. [Custom Analysis Pipeline](#4-custom-analysis-pipeline)

In [ ]:
from datetime import date, timedelta
import pandas as pd
import matplotlib.pyplot as plt

# Import example analysis functions
from dlt_ibapi.strategies import (
    calculate_option_metrics,
    find_liquid_contracts,
    analyze_volume_patterns,
)

# Import data readers
from dlt_ibapi.repositories import (
    OptionBarsReader,
    OptionChainSnapshotReader,
    EquityBarsReader,
)

print("✓ Imports complete")

## 1. Calculate Option Metrics

Calculate basic metrics (volume, spread, price range) for a specific option contract.

In [ ]:
# Example: Calculate metrics for an AAPL call option
metrics = calculate_option_metrics(
    database_path="../data",
    underlying="AAPL",
    expiry=date(2025, 12, 20),
    strike=180.0,
    right="C",
    start_date=date(2025, 11, 1),
    end_date=date(2025, 11, 15),
    bar_size="5 mins",
)

if metrics:
    print(f"\n📊 Metrics for AAPL {metrics.strike}{metrics.right} expiring {metrics.expiry}:")
    print(f"  Average volume: {metrics.avg_volume:,.0f}")
    print(f"  Average spread: {metrics.avg_spread:.2%}")
    print(f"  Price range: ${metrics.price_range:.2f}")
else:
    print("\n⚠️  No data found for this contract")
    print("  Tip: Make sure you've backfilled option bars first using:")
    print("       dlt-ibapi backfill-options AAPL 180.0 --mode atm")

## 2. Find Liquid Contracts

Filter option chains to find contracts meeting minimum liquidity criteria.

In [ ]:
# Example: Find liquid AAPL options
liquid_contracts = find_liquid_contracts(
    database_path="../data",
    underlying="AAPL",
    as_of=date(2025, 11, 15),
    min_volume=100.0,       # Min average daily volume
    max_spread_pct=0.05,    # Max 5% bid-ask spread
    min_dte=7,              # Min 7 days to expiration
    max_dte=60,             # Max 60 days to expiration
)

if not liquid_contracts.empty:
    print(f"\n✓ Found {len(liquid_contracts)} liquid contracts")
    print("\nTop 10 by volume:")
    print(liquid_contracts.nlargest(10, 'avg_volume')[[
        'strike', 'right', 'dte', 'avg_volume', 'avg_spread_pct'
    ]].to_string(index=False))
else:
    print("\n⚠️  No liquid contracts found matching criteria")
    print("  Tip: Make sure you've captured option chain snapshots using:")
    print("       dlt-ibapi snapshot AAPL --min-dte 7 --max-dte 60")

## 3. Analyze Volume Patterns

Analyze intraday volume patterns for an option contract.

In [ ]:
# Example: Analyze volume patterns
volume_analysis = analyze_volume_patterns(
    database_path="../data",
    underlying="AAPL",
    expiry=date(2025, 12, 20),
    strike=180.0,
    right="C",
    start_date=date(2025, 11, 1),
    end_date=date(2025, 11, 15),
    bar_size="5 mins",
)

if not volume_analysis.empty:
    print("\n📈 Daily Volume Statistics:")
    print(volume_analysis)
    
    # Plot volume over time
    plt.figure(figsize=(12, 6))
    plt.bar(range(len(volume_analysis)), volume_analysis['volume_sum'])
    plt.title('Daily Volume for AAPL 180C', fontsize=14, fontweight='bold')
    plt.xlabel('Trading Day', fontsize=12)
    plt.ylabel('Total Volume', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️  No volume data found")

## 4. Custom Analysis Pipeline

Build a custom analysis pipeline using the data readers directly.

In [ ]:
# Example: Custom analysis combining multiple data sources

# 1. Initialize readers
equity_reader = EquityBarsReader(database_path="../data", dataset_name="stocks")
option_reader = OptionBarsReader(database_path="../data", dataset_name="options")

# 2. Get underlying equity data
equity_bars = equity_reader.get_bars(
    symbol="AAPL",
    bar_size="1 day",
    start_date=date(2025, 11, 1),
    end_date=date(2025, 11, 15),
)

print(f"\n✓ Loaded {len(equity_bars)} equity bars")

# 3. Get option data for same period
option_bars = option_reader.get_bars(
    underlying="AAPL",
    expiry=date(2025, 12, 20),
    strike=180.0,
    right="C",
    bar_size="1 day",
    start_date=date(2025, 11, 1),
    end_date=date(2025, 11, 15),
)

print(f"✓ Loaded {len(option_bars)} option bars")

# 4. Combine and analyze
if not equity_bars.empty and not option_bars.empty:
    # Merge on date
    equity_bars['date'] = pd.to_datetime(equity_bars['time']).dt.date
    option_bars['date'] = pd.to_datetime(option_bars['time']).dt.date
    
    combined = pd.merge(
        equity_bars[['date', 'close']].rename(columns={'close': 'stock_price'}),
        option_bars[['date', 'close']].rename(columns={'close': 'option_price'}),
        on='date',
        how='inner'
    )
    
    # Calculate simple metrics
    combined['option_pct_of_stock'] = (combined['option_price'] / combined['stock_price']) * 100
    
    print("\n📊 Combined Analysis:")
    print(combined[['date', 'stock_price', 'option_price', 'option_pct_of_stock']])
    
    # Plot both on same chart (dual axis)
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel('Stock Price ($)', color='blue', fontsize=12)
    ax1.plot(combined.index, combined['stock_price'], color='blue', marker='o', label='Stock')
    ax1.tick_params(axis='y', labelcolor='blue')
    
    ax2 = ax1.twinx()
    ax2.set_ylabel('Option Price ($)', color='red', fontsize=12)
    ax2.plot(combined.index, combined['option_price'], color='red', marker='s', label='Option')
    ax2.tick_params(axis='y', labelcolor='red')
    
    plt.title('AAPL Stock vs 180C Option Price', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()
else:
    print("\n⚠️  Insufficient data for analysis")

## Summary

This notebook demonstrated:

1. ✓ Calculating option metrics using pre-built functions
2. ✓ Filtering option chains for liquid contracts
3. ✓ Analyzing volume patterns over time
4. ✓ Building custom analysis pipelines with data readers

### Key Takeaways

- **Data Readers**: Use `EquityBarsReader`, `OptionBarsReader`, and `OptionChainSnapshotReader` to query data
- **Parquet Storage**: All queries run against Parquet files with predicate pushdown for fast filtering
- **Composable**: Build complex analyses by combining simple, testable functions
- **Type-Safe**: Pydantic models ensure data integrity

### Next Steps

- Explore the `src/dlt_ibapi/strategies/example_analysis.py` source code
- Build your own analysis functions following the same patterns
- Check out the Reader API documentation in `docs/API_REFERENCE.md`